# Notebook 06 — Feature-Selektion für SVM Stufe 2

**Frage:** Verbessert sich die SVM-Feinklassifikation (Apfel / Kaugummi / Skyr / Essen)
wenn wir weniger, aber relevantere Features verwenden?

**Methoden:**

| Methode | Modell | Idee |
|---------|--------|------|
| Permutation Importance (PI) | RBF-SVM | Schädliche Features (PI < 0) entfernen |
| RFECV | LinearSVC | Minimale Feature-Menge via Rekursiver Elimination |
| Accuracy-Kurve | RBF-SVM | LOO-Accuracy vs. Anzahl Features (PI-Ranking) |

**Daten & Vorverarbeitung** identisch zu NB04/05 (k=5 ME, 25-s-Fenster).

---
## 1. Setup

In [ ]:
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch

from sklearn.svm import SVC, LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, StratifiedKFold
from sklearn.feature_selection import RFECV
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, classification_report

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

CLASSES_RAW    = ['Apfel', 'Kaugummi', 'Skyr', 'Still', 'Essen']
CLASSES_FINE   = ['Apfel', 'Kaugummi', 'Skyr', 'Essen']
TO_COARSE      = {'Apfel': 'Essen', 'Kaugummi': 'Essen',
                  'Skyr':  'Essen', 'Still':    'Still', 'Essen': 'Essen'}
COLORS = {'Apfel': '#e15759', 'Kaugummi': '#4e79a7', 'Skyr': '#f28e2b', 'Essen': '#b07aa1'}

FS, TRIM_SECS, WINDOW_SECS, MIN_TAIL_SECS, K = 50.0, 2, 25.0, 20.0, 5

SVM_PARAMS = dict(kernel='rbf', C=10, probability=True,
                  class_weight='balanced', random_state=42)

# NB05-Referenz: SVM mit allen 36 Features
NB05_SVM_S2  = None   # wird nach dem ersten LOO-Lauf gesetzt
NB05_SVM_E2E = 0.0    # Platzhalter

---
## 2. Daten laden, segmentieren, Features extrahieren

In [ ]:
DATA_DIR = Path('../data/raw')
_SKIP    = {'Metadata.csv', 'Annotation.csv'}

def preprocess(df):
    df = df.copy()
    t  = df['seconds_elapsed']
    df = df[(t >= t.iloc[0] + TRIM_SECS) &
            (t <= t.iloc[-1] - TRIM_SECS)].reset_index(drop=True)
    df['lin_x']     = df['accelerationX']
    df['lin_y']     = df['accelerationY']
    df['lin_z']     = df['accelerationZ']
    df['magnitude'] = np.sqrt(df['lin_x']**2 + df['lin_y']**2 + df['lin_z']**2)
    return df

def split_windows(df):
    t = df['seconds_elapsed'].values
    t_start, t_end = t[0], t[-1]
    out = []
    while t_start + MIN_TAIL_SECS <= t_end:
        t_stop = t_start + WINDOW_SECS
        w = df[(t >= t_start) & (t < t_stop)].reset_index(drop=True)
        if len(w) > 1 and (w['seconds_elapsed'].iloc[-1] - w['seconds_elapsed'].iloc[0]) >= MIN_TAIL_SECS:
            out.append(w)
        t_start = t_stop
    return out

def movement_mask(df):
    thr = max(0.02, K * df['magnitude'].median())
    return df['magnitude'].rolling(50, center=True, min_periods=1).max() <= thr

def extract_features(df):
    feats = {}
    for col in ['lin_x', 'lin_y', 'lin_z', 'magnitude']:
        feats[f'{col}_mean'] = df[col].mean()
        feats[f'{col}_std']  = df[col].std()
        feats[f'{col}_max']  = df[col].abs().max()
    feats['stillness_ratio'] = (df['magnitude'] < 0.02).mean()
    feats['movement_events'] = int((df['magnitude'] > df['magnitude'].quantile(0.75)).sum())
    for col in ['rotationRateX', 'rotationRateY', 'rotationRateZ']:
        feats[f'{col}_mean'] = df[col].mean()
        feats[f'{col}_std']  = df[col].std()
        feats[f'{col}_max']  = df[col].abs().max()
    for col in ['pitch', 'roll', 'yaw']:
        feats[f'{col}_mean']  = df[col].mean()
        feats[f'{col}_std']   = df[col].std()
        feats[f'{col}_range'] = df[col].max() - df[col].min()
    nperseg = min(256, len(df) // 2)
    freqs, psd = welch(df['magnitude'].values, fs=FS, nperseg=nperseg)
    chew = (freqs >= 0.5) & (freqs <= 4.0)
    cf, cp = freqs[chew], psd[chew]
    feats['total_power']        = float(psd.sum())
    feats['chew_band_power']    = float(cp.sum())
    feats['rhythmicity']        = feats['chew_band_power'] / feats['total_power'] if feats['total_power'] > 0 else 0.0
    feats['dominant_chew_freq'] = float(cf[np.argmax(cp)]) if len(cp) > 0 else 0.0
    return feats

# ── Laden ─────────────────────────────────────────────────────────────────────
sessions = {cls: [] for cls in CLASSES_RAW}
for zf in sorted(DATA_DIR.glob('*.zip')):
    for cls in CLASSES_RAW:
        if zf.name.startswith(cls + '_'):
            sessions[cls].append(zf); break

rows, y_raw = [], []
for cls in CLASSES_RAW:
    for zf in sessions[cls]:
        with zipfile.ZipFile(zf) as z:
            csv_name = next(f for f in z.namelist() if f.endswith('.csv') and f not in _SKIP)
            with z.open(csv_name) as f:
                df = preprocess(pd.read_csv(f))
        for w in split_windows(df):
            clean = w[movement_mask(w)].reset_index(drop=True)
            if len(clean) > 50:
                rows.append(extract_features(clean))
                y_raw.append(cls)

X_all    = pd.DataFrame(rows)
y_raw    = np.array(y_raw)
y_coarse = np.array([TO_COARSE[c] for c in y_raw])
eat_mask = y_coarse == 'Essen'
X_eat    = X_all[eat_mask].reset_index(drop=True)
y_eat    = y_raw[eat_mask]

print(f'Essen-Samples: {len(y_eat)}')
for cls in CLASSES_FINE:
    print(f'  {cls:10s}: {(y_eat==cls).sum()}')
print(f'Features: {X_eat.shape[1]}')

---
## 3. Baseline — SVM mit allen 36 Features (LOO-CV)

In [ ]:
def svm_loo(X_np, y):
    yt, yp = [], []
    for tr, te in LeaveOneOut().split(X_np):
        sc   = StandardScaler()
        Xtr  = sc.fit_transform(X_np[tr])
        Xte  = sc.transform(X_np[te])
        clf  = SVC(**SVM_PARAMS)
        clf.fit(Xtr, y[tr])
        yp.append(clf.predict(Xte)[0])
        yt.append(y[te][0])
    return np.array(yt), np.array(yp)

print(f'SVM Baseline LOO ({len(y_eat)} Folds, 36 Features)…')
yt_base, yp_base = svm_loo(X_eat.values, y_eat)
acc_base = accuracy_score(yt_base, yp_base)
NB05_SVM_S2 = acc_base

print(f'\nSVM Stufe 2 — alle 36 Features:  {acc_base:.1%}')
print()
print(classification_report(yt_base, yp_base, labels=CLASSES_FINE, zero_division=0))

---
## 4. Permutation Importance (RBF-SVM)

Trainiert auf dem vollen Datensatz. Features mit **mittlerer PI < 0** schaden der Genauigkeit
und werden als Kandidaten zum Entfernen markiert.

In [ ]:
sc_full = StandardScaler()
X_eat_sc = sc_full.fit_transform(X_eat.values)

svm_pi = SVC(**SVM_PARAMS)
svm_pi.fit(X_eat_sc, y_eat)

print('Permutation Importance berechnen (15 Wiederholungen)…')
pi = permutation_importance(svm_pi, X_eat_sc, y_eat,
                             n_repeats=15, random_state=42, n_jobs=-1)

pi_mean = pd.Series(pi.importances_mean, index=X_eat.columns)
pi_std  = pd.Series(pi.importances_std,  index=X_eat.columns)

harmful  = pi_mean[pi_mean < 0].index.tolist()
neutral  = pi_mean[(pi_mean >= 0) & (pi_mean < 0.002)].index.tolist()
useful   = pi_mean[pi_mean >= 0.002].index.tolist()
FEATS_PI = [f for f in X_eat.columns if f not in harmful]

print(f'\nErgebnis:')
print(f'  Schädlich  (PI < 0):     {len(harmful):2d}  → werden entfernt')
print(f'  Neutral  (0 ≤ PI < 0.002): {len(neutral):2d}')
print(f'  Nützlich (PI ≥ 0.002):   {len(useful):2d}')
print(f'  Behalten insgesamt:      {len(FEATS_PI)}')
if harmful:
    print(f'\nEntfernte Features: {harmful}')

In [ ]:
order  = pi_mean.sort_values(ascending=True)
colors = ['#c00000' if v < 0 else ('#cccccc' if v < 0.002 else '#4e79a7')
          for v in order.values]

fig, ax = plt.subplots(figsize=(8, 10))
ax.barh(range(len(order)), order.values,
        xerr=pi_std[order.index].values,
        color=colors, edgecolor='white', capsize=2, height=0.7)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order.index, fontsize=9)
ax.axvline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax.set_xlabel('Δ Accuracy bei Permutation')
ax.set_title('Permutation Importance — SVM Stufe 2\n'
             'Rot = entfernt  |  Grau = neutral  |  Blau = behalten',
             fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/images/nb06_pi_svm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print(f'SVM LOO mit PI-bereinigten Features ({len(FEATS_PI)})…')
yt_pi, yp_pi = svm_loo(X_eat[FEATS_PI].values, y_eat)
acc_pi = accuracy_score(yt_pi, yp_pi)

print(f'\nSVM Stufe 2 — alle 36:       {acc_base:.1%}')
print(f'SVM Stufe 2 — PI-bereinigt ({len(FEATS_PI):2d}): {acc_pi:.1%}  ({acc_pi - acc_base:+.1%})')

---
## 5. RFECV — Rekursive Feature-Elimination

**LinearSVC** als Basis (RBF-SVM hat kein `coef_`, RFECV benötigt Feature-Gewichte).
Die selektierten Features werden dann mit **RBF-SVM** via LOO evaluiert.

In [ ]:
sc_rfe = StandardScaler()
X_rfe  = sc_rfe.fit_transform(X_eat.values)

lsvc = LinearSVC(C=1.0, class_weight='balanced', max_iter=2000, random_state=42)
rfe  = RFECV(lsvc, step=1, cv=StratifiedKFold(5), scoring='accuracy', n_jobs=-1)

print('RFECV läuft (LinearSVC, 5-fold CV)…')
rfe.fit(X_rfe, y_eat)

FEATS_RFE = X_eat.columns[rfe.support_].tolist()
print(f'\nRFECV-Ergebnis: {rfe.n_features_} Features optimal')
print(f'Selektierte Features: {FEATS_RFE}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
n_feat = np.arange(1, len(rfe.cv_results_['mean_test_score']) + 1)
mean_s = rfe.cv_results_['mean_test_score']
std_s  = rfe.cv_results_['std_test_score']
ax.plot(n_feat, mean_s, 'o-', color='#4e79a7', linewidth=2, markersize=5)
ax.fill_between(n_feat, mean_s - std_s, mean_s + std_s, alpha=0.2, color='#4e79a7')
ax.axvline(rfe.n_features_, color='#e15759', linestyle='--',
           label=f'Optimal: {rfe.n_features_} Features')
ax.axhline(acc_base, color='grey', linestyle=':', label=f'Baseline (36 Features): {acc_base:.1%}')
ax.set_xlabel('Anzahl Features')
ax.set_ylabel('CV-Accuracy (5-fold)')
ax.set_title('RFECV — LinearSVC als Proxy für RBF-SVM', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/images/nb06_rfecv_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print(f'SVM LOO mit RFECV-Features ({len(FEATS_RFE)})…')
yt_rfe, yp_rfe = svm_loo(X_eat[FEATS_RFE].values, y_eat)
acc_rfe = accuracy_score(yt_rfe, yp_rfe)

print(f'\nSVM Stufe 2 — alle 36:       {acc_base:.1%}')
print(f'SVM Stufe 2 — PI-bereinigt ({len(FEATS_PI):2d}): {acc_pi:.1%}  ({acc_pi - acc_base:+.1%})')
print(f'SVM Stufe 2 — RFECV ({len(FEATS_RFE):2d}):       {acc_rfe:.1%}  ({acc_rfe - acc_base:+.1%})')

---
## 6. LOO-Accuracy vs. Anzahl Features

Features werden nach **PI-Ranking** (absteigend nützlich) schrittweise hinzugefügt.
Zeigt den echten LOO-Effekt — nicht nur CV-Schätzung.

In [ ]:
# Features sortiert von nützlichsten zu schädlichsten (nach PI-Mean)
feat_order = pi_mean.sort_values(ascending=False).index.tolist()

# LOO für wachsende Feature-Mengen (Schritte von 1)
step_sizes = list(range(1, len(feat_order) + 1))
accs_curve = []

print(f'LOO-Kurve berechnen ({len(step_sizes)} Punkte × {len(y_eat)} Folds)…')
for k in step_sizes:
    feats_k = feat_order[:k]
    yt_k, yp_k = svm_loo(X_eat[feats_k].values, y_eat)
    accs_curve.append(accuracy_score(yt_k, yp_k))
    if k % 5 == 0 or k == 1:
        print(f'  k={k:2d}: {accs_curve[-1]:.1%}')

best_k   = int(np.argmax(accs_curve)) + 1
best_acc = max(accs_curve)
FEATS_BEST_K = feat_order[:best_k]
print(f'\nBestes k: {best_k} Features → {best_acc:.1%}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(step_sizes, accs_curve, 'o-', color='#4e79a7',
        linewidth=2, markersize=4, alpha=0.8, label='LOO-Accuracy')
ax.axvline(best_k,    color='#e15759', linestyle='--',
           label=f'Bestes k={best_k}: {best_acc:.1%}')
ax.axvline(len(FEATS_RFE), color='#f28e2b', linestyle=':',
           label=f'RFECV: k={len(FEATS_RFE)}: {acc_rfe:.1%}')
ax.axhline(acc_base, color='grey', linestyle=':', alpha=0.7,
           label=f'Baseline 36 Features: {acc_base:.1%}')
ax.set_xlabel('Anzahl Features (nach PI-Ranking sortiert)')
ax.set_ylabel('LOO-Accuracy (Stufe 2)')
ax.set_title('SVM Stufe 2 — LOO-Accuracy vs. Anzahl Features', fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0.75, 1.0)
plt.tight_layout()
plt.savefig('../reports/images/nb06_accuracy_curve.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Ergebnisübersicht

In [ ]:
print('╔══════════════════════════════════════════════════════════╗')
print('║     SVM Stufe 2 — Feature-Selektion Vergleich           ║')
print('╠══════════════════════════════════════════════════════════╣')
variants = [
    ('Alle Features',      36,            acc_base),
    ('PI-bereinigt',       len(FEATS_PI), acc_pi),
    ('RFECV (LinearSVC)',  len(FEATS_RFE),acc_rfe),
    (f'Beste k (LOO)',     best_k,        best_acc),
]
for name, n, acc in variants:
    delta = acc - acc_base
    star  = ' ← Empfehlung' if acc == max(v[2] for v in variants) else ''
    print(f'║  {name:22s}  {n:2d} Features  {acc:.1%}  ({delta:+.1%}){star}')
print('╚══════════════════════════════════════════════════════════╝')

print(f'\nEmpfohlene Features ({best_k} Stück):')
for f in FEATS_BEST_K:
    print(f'  {f}')

In [ ]:
# Klassifikationsreport für beste Variante
yt_best, yp_best = svm_loo(X_eat[FEATS_BEST_K].values, y_eat)
print(f'Detail-Report — SVM mit {best_k} Features (bestes LOO-k):')
print(classification_report(yt_best, yp_best, labels=CLASSES_FINE, zero_division=0))